# Visualizing Graphs with Mermaid

A graph you cannot see is a graph you cannot debug. SynapseKit can render any compiled `StateGraph` as a Mermaid flowchart — a plain-text diagram format supported by GitHub, Notion, Confluence, and dozens of other tools. You can also export PNG images directly from Python.

**What you'll build:** A visualization toolkit that renders a multi-branch graph as a Mermaid string, highlights the active execution thread, and exports a PNG image.

**Time:** ~10 min | **Difficulty:** Beginner

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set. For PNG export, install matplotlib.

**What you'll learn:**
- Call `get_mermaid()` on a compiled graph to get the diagram string
- Use `to_mermaid(highlight_thread=)` to highlight a specific run's execution path
- Use `GraphVisualizer` for rich rendering and PNG export
- Customize node shapes, edge labels, and diagram direction

In [ ]:
!pip install synapsekit -q
# Optional: for PNG export
# !pip install matplotlib -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Build a Graph to Visualize

A review sentiment graph with conditional routing — classify, then branch to positive/negative/neutral handlers with a possible escalation path.

In [ ]:
from __future__ import annotations
import asyncio
from dataclasses import dataclass

from synapsekit.graph import StateGraph, CompiledGraph
from synapsekit.graph.visualizer import GraphVisualizer
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

llm = OpenAILLM(model="gpt-4o-mini", config=LLMConfig(temperature=0.3))

@dataclass
class ReviewState:
    text: str
    sentiment: str  = ""    # "positive" | "negative" | "neutral"
    response: str   = ""
    escalated: bool = False


async def classify(state: ReviewState) -> ReviewState:
    result = await llm.agenerate(
        f"Classify this review as positive, negative, or neutral. "
        f"Reply with only the word.\n\nReview: {state.text}"
    )
    state.sentiment = result.text.strip().lower()
    return state


async def handle_positive(state: ReviewState) -> ReviewState:
    result = await llm.agenerate(f"Write a brief thank-you response to: {state.text}")
    state.response = result.text
    return state


async def handle_negative(state: ReviewState) -> ReviewState:
    result = await llm.agenerate(
        f"Write an empathetic apology and offer to help for: {state.text}"
    )
    state.response = result.text
    return state


async def handle_neutral(state: ReviewState) -> ReviewState:
    result = await llm.agenerate(f"Write a friendly acknowledgment for: {state.text}")
    state.response = result.text
    return state


async def escalate(state: ReviewState) -> ReviewState:
    state.escalated = True
    state.response = "This review has been escalated to a customer success manager."
    return state


def route_sentiment(state: ReviewState) -> str:
    return state.sentiment if state.sentiment in ("positive", "negative", "neutral") else "neutral"


def needs_escalation(state: ReviewState) -> str:
    # Escalate if the negative response is very short (couldn't generate a full reply)
    return "escalate" if state.escalated or len(state.response) < 20 else "done"


def build_graph() -> CompiledGraph:
    graph = StateGraph(ReviewState)

    graph.add_node("classify",         classify)
    graph.add_node("handle_positive",  handle_positive)
    graph.add_node("handle_negative",  handle_negative)
    graph.add_node("handle_neutral",   handle_neutral)
    graph.add_node("escalate",         escalate)

    graph.set_entry_point("classify")

    graph.add_conditional_edges(
        "classify",
        route_sentiment,
        {
            "positive": "handle_positive",
            "negative": "handle_negative",
            "neutral":  "handle_neutral",
        }
    )

    graph.add_conditional_edges(
        "handle_negative",
        needs_escalation,
        {
            "escalate": "escalate",
            "done":     "__end__",
        }
    )

    return graph.compile()

## Step 2: Generate a Mermaid Diagram

`get_mermaid()` returns the full Mermaid flowchart as a string. Paste it into https://mermaid.live to view the rendered diagram.

In [ ]:
def print_mermaid(compiled: CompiledGraph) -> None:
    # get_mermaid() returns the full Mermaid flowchart as a string.
    # Paste it into https://mermaid.live to view the rendered diagram.
    mermaid_src = compiled.get_mermaid()
    print("=== MERMAID DIAGRAM ===")
    print(mermaid_src)

compiled = build_graph()
print_mermaid(compiled)

## Step 3: Highlight a Specific Execution Thread

`to_mermaid(highlight_thread=run_id)` marks the nodes visited by a specific run in a different color.

In [ ]:
from synapsekit.graph.checkpointing import SQLiteCheckpointer

async def run_and_highlight(text: str, run_id: str) -> None:
    # Use a checkpointer so thread data is available for highlighting
    checkpointer = SQLiteCheckpointer(db_path="./viz_checkpoints.db")

    graph = StateGraph(ReviewState, checkpointer=checkpointer)

    # Re-register all nodes and edges
    graph.add_node("classify",         classify)
    graph.add_node("handle_positive",  handle_positive)
    graph.add_node("handle_negative",  handle_negative)
    graph.add_node("handle_neutral",   handle_neutral)
    graph.add_node("escalate",         escalate)
    graph.set_entry_point("classify")
    graph.add_conditional_edges(
        "classify", route_sentiment,
        {"positive": "handle_positive", "negative": "handle_negative", "neutral": "handle_neutral"}
    )
    graph.add_conditional_edges(
        "handle_negative", needs_escalation,
        {"escalate": "escalate", "done": "__end__"}
    )
    compiled_cp = graph.compile()

    await compiled_cp.arun(ReviewState(text=text), run_id=run_id)

    # to_mermaid(highlight_thread=run_id) marks visited nodes in a different color
    highlighted = compiled_cp.to_mermaid(
        highlight_thread=run_id,
        highlight_color="#a8d8a8",  # Light green for visited nodes
    )
    print("\n=== HIGHLIGHTED THREAD ===")
    print(highlighted)

asyncio.run(run_and_highlight(
    "The product broke after one day and customer service ignored my emails.",
    "review-001"
))

## Step 4: Export a PNG Image

`GraphVisualizer` renders the Mermaid diagram using matplotlib and writes the image to a file.

In [ ]:
def export_png(compiled: CompiledGraph, output_path: str = "./review_graph.png") -> None:
    """Render the graph as a PNG using GraphVisualizer.

    Requires matplotlib: pip install matplotlib
    """
    try:
        viz = GraphVisualizer(compiled)

        # save() renders the Mermaid diagram using matplotlib and writes the image
        viz.save(output_path)
        print(f"Graph image saved to {output_path}")

    except ImportError:
        print("Install matplotlib for PNG export: pip install matplotlib")
        print("Falling back to Mermaid string output:")
        print(compiled.get_mermaid())

export_png(compiled)

## Step 5: Customize the Diagram

Pass options to `get_mermaid()` for more control over the output.

In [ ]:
# Include condition labels on edges
print("=== WITH CONDITION LABELS ===")
labeled = compiled.get_mermaid(include_conditions=True)
print(labeled)

# Change diagram direction (TD = top-down, LR = left-right)
print("\n=== LEFT-TO-RIGHT LAYOUT ===")
horizontal = compiled.get_mermaid(direction="LR")
print(horizontal)

# Use different node shapes
print("\n=== CUSTOM NODE SHAPES ===")
shaped = compiled.get_mermaid(
    node_shapes={
        "classify":        "diamond",   # Decision node
        "handle_positive": "rounded",   # Default nodes are rectangles
        "escalate":        "hexagon",
    }
)
print(shaped)

## Complete Working Example

Build the graph, print a plain diagram, print one with condition labels, run a sample, highlight the thread, and export a PNG.

In [ ]:
async def main():
    compiled = build_graph()

    # 1. Plain Mermaid string
    print_mermaid(compiled)

    # 2. Mermaid with condition labels
    print("\n=== WITH CONDITION LABELS ===")
    print(compiled.get_mermaid(include_conditions=True))

    # 3. Run a sample and highlight the execution path
    print("\n=== HIGHLIGHTED THREAD ===")
    checkpointer = SQLiteCheckpointer(db_path="./viz_checkpoints.db")
    graph_with_cp = StateGraph(ReviewState, checkpointer=checkpointer)
    graph_with_cp.add_node("classify",         classify)
    graph_with_cp.add_node("handle_positive",  handle_positive)
    graph_with_cp.add_node("handle_negative",  handle_negative)
    graph_with_cp.add_node("handle_neutral",   handle_neutral)
    graph_with_cp.add_node("escalate",         escalate)
    graph_with_cp.set_entry_point("classify")
    graph_with_cp.add_conditional_edges(
        "classify", route_sentiment,
        {"positive": "handle_positive", "negative": "handle_negative", "neutral": "handle_neutral"}
    )
    graph_with_cp.add_conditional_edges(
        "handle_negative", needs_escalation,
        {"escalate": "escalate", "done": "__end__"}
    )
    compiled_cp = graph_with_cp.compile()

    review = "The product broke after one day and customer service ignored my emails."
    await compiled_cp.arun(ReviewState(text=review), run_id="review-main-001")

    highlighted = compiled_cp.to_mermaid(highlight_thread="review-main-001")
    print(highlighted)

    # 4. Export PNG
    export_png(compiled)

asyncio.run(main())

## Embedding in Docusaurus

Docusaurus supports Mermaid diagrams natively in MDX files. Use a fenced code block with `mermaid` as the language:

````md
```mermaid
graph TD
  classify -->|positive| handle_positive
  classify -->|negative| handle_negative
  handle_negative --> escalate
```
````

You can also generate the Mermaid string programmatically and write it directly into a `.md` file as part of your documentation build pipeline.

## Next Steps

- **[Linear Workflow](linear-workflow.ipynb)** — build the simplest graph to have something to visualize
- **[Human-in-the-Loop](human-in-the-loop.ipynb)** — visualize a graph with interrupt nodes
- **[Conditional Routing](conditional-routing.ipynb)** — understand the conditional edges shown in the diagram